# Tutorial 11-3: Compact Representations - Maximal and Closed Itemsets
**Course:** CSEN 140: Data Mining/Machine Learning  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
In many applications, the number of frequent itemsets is too large to be useful. Some itemsets are redundant because they have identical support as their supersets. This tutorial demonstrates how to find **Maximal** and **Closed** frequent itemsets to provide a compact representation of the data.

### Key Definitions
1. **Maximal Frequent Itemset:** An itemset is maximal frequent if it is frequent and **none** of its immediate supersets are frequent.
2. **Closed Frequent Itemset:** An itemset is closed if it is frequent and **none** of its immediate supersets have the same support count. All extensions of a closed itemset must have a lower support.
3. **Relationship:** All Maximal frequent itemsets are Closed, and all Closed frequent itemsets are Frequent.

In [ ]:
# Using the dataset and support counts from the lecture slides
transactions = [
    {'A', 'B', 'C'},       # TID 1
    {'A', 'B', 'C', 'D'},  # TID 2
    {'B', 'C', 'E'},       # TID 3
    {'A', 'C', 'D', 'E'},  # TID 4
    {'D', 'E'}             # TID 5
]

minsup_count = 2
print(f"Minimum Support Count: {minsup_count}")

### 1. Identify All Frequent Itemsets
First, we calculate the support for every possible subset and filter by our threshold.

In [ ]:
from itertools import chain, combinations

def get_all_subsets(items):
    return chain.from_iterable(combinations(items, r) for r in range(1, len(items) + 1))

# Get all unique items
unique_items = sorted(list(set().union(*transactions)))

# Calculate support for ALL possible itemsets
support_counts = {}
for subset in get_all_subsets(unique_items):
    s = frozenset(subset)
    count = sum(1 for t in transactions if s.issubset(t))
    if count >= minsup_count:
        support_counts[s] = count

print(f"Found {len(support_counts)} frequent itemsets.")

### 2. Identifying Maximal Frequent Itemsets
We check if any superset of a frequent itemset is also frequent. If not, it is **Maximal**.

In [ ]:
maximal_itemsets = []

for itemset in support_counts:
    is_maximal = True
    # Check all other frequent itemsets to see if one is a superset
    for other in support_counts:
        if itemset != other and itemset.issubset(other):
            is_maximal = False
            break
    if is_maximal:
        maximal_itemsets.append(itemset)

print(f"Maximal Frequent Itemsets ({len(maximal_itemsets)}):")
for m in maximal_itemsets: print(set(m))

### 3. Identifying Closed Frequent Itemsets
We check if any superset has the **same** support count. If every superset has a lower count, the itemset is **Closed**.

In [ ]:
closed_itemsets = []

for itemset, count in support_counts.items():
    is_closed = True
    for other, other_count in support_counts.items():
        # Check if 'other' is a superset with the SAME support
        if itemset != other and itemset.issubset(other) and count == other_count:
            is_closed = False
            break
    if is_closed:
        closed_itemsets.append((itemset, count))

print(f"Closed Frequent Itemsets ({len(closed_itemsets)}):")
for c, count in closed_itemsets: print(f"{set(c)}: count={count}")

### 4. Comparison Table
Let's compare the count of these itemsets to see the compression achieved.

In [ ]:
import pandas as pd

summary = {
    "Type": ["Frequent", "Closed", "Maximal"],
    "Count": [len(support_counts), len(closed_itemsets), len(maximal_itemsets)]
}
print(pd.DataFrame(summary))

## Summary & Conclusion

Compact representations solve the 'pattern explosion' problem:
1. **Maximal Itemsets** provide the most compression but lose support information for their subsets. They effectively define the 'boundary' between frequent and infrequent patterns.
2. **Closed Itemsets** provide a 'lossless' compression. You can determine the support of any frequent itemset just by looking at the set of closed itemsets.
3. **Practical Use:** When mining millions of rules, focusing only on Closed or Maximal patterns can reduce the output size by orders of magnitude while still providing a complete picture of the underlying associations.
